ReACT Pattern

In [17]:
import os
import openai
import json
import io

from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [18]:
load_dotenv(override=True)

True

In [19]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if openai_api_key:
    print(f"OpenAI API key exists and begins with: " + openai_api_key[:6] + "...")
else:
    print("OpenAI API key does not exist")

if anthropic_api_key:
    print(f"Anthropic API key exists and begins with: " + anthropic_api_key[:6] + "...")
else:
    print("Anthropic API key does not exist")

if google_api_key:
    print(f"Google API key exists and begins with: " + google_api_key[:6] + "...")
else:
    print("Google API key does not exist")

OpenAI API key exists and begins with: sk-pro...
Anthropic API key exists and begins with: sk-ant...
Google API key exists and begins with: AIzaSy...


In [26]:
from openai import OpenAI

openai = OpenAI()

# Request prompt
request = (
    "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
    "Answer only with the question, no explanation or other text."
)


# Function to generate a question
def generate_question(prompt: str) -> str:
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    question = response.choices[0].message.content
    if question is None:
        raise ValueError("No response content received from OpenAI")
    return question

def react_agent_decide_model(question: str) -> str:
    prompt = f"""
            You are an intelligent AI assistant tasked with evaluating which language model is most suitable to answer a given question.

            Available models:
            - OpenAI: excels at reasoning and factual answers.
            - Claude: better for philosophical, nuanced, and ethical topics.
            - Gemini: good for concise and structured summaries.
            
            Here is the question to answer:
            "{question}"

            ### Thought:
            Which model is best suited to answer this question, and why?

            ### Action:
            Respond with only the model name you choose (e.g., "Claude").
            """
    response = openai.chat.completions.create(
        model="o3-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    model = response.choices[0].message.content
    if model is None:
        raise ValueError("No response content received from OpenAI")
    return model.strip()

# Function to generate an answer using OpenAI
def generate_openai_answer(prompt):
    answer = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    ).choices[0].message.content
    if answer is None:
        raise ValueError("No response content received from OpenAI")
    return answer

# Function to generate an answer using Anthropic
def generate_anthropic_answer(prompt):
    model_name = "claude-3-7-sonnet-latest"
    claude = Anthropic()
    answer = claude.messages.create(
        model=model_name, 
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1000
    ).content[0].text
    return answer

# Function to generate an answer using Gemini
def generate_gemini_answer(prompt):
    gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    model_name = "gemini-2.0-flash"
    answer = gemini.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}]
    ).choices[0].message.content
    if answer is None:
        raise ValueError("No response content received from Gemini")
    return answer


def main():
    print("Generating question...")
    question = generate_question(request)
    print(f"\n🧠 Question: {question}\n")

    selected_model = react_agent_decide_model(question)
    print(f"\n🤖 Selected model: {selected_model}\n")

    if selected_model.lower() == "openai":
        answer = generate_openai_answer(question)
    elif selected_model.lower() == "claude":
        answer = generate_anthropic_answer(question)
    elif selected_model.lower() == "gemini":
        answer = generate_gemini_answer(question)
    else:
        raise ValueError(f"Unsupported model: {selected_model}")

    print(f"\n🤖 Answer: {answer}\n")





In [28]:
main()

Generating question...

🧠 Question: If you could redesign the structure of society to prioritize well-being and sustainability over profit, what specific changes would you propose in the domains of education, economics, and governance, and how would you address potential resistance to these changes?


🤖 Selected model: Claude


🤖 Answer: # Reimagining Society for Well-being and Sustainability

## Education Domain
- Shift from standardized testing to competency-based learning that emphasizes critical thinking, emotional intelligence, and systems thinking
- Integrate sustainability literacy and practical life skills throughout curriculum
- Implement universal access to education with personalized learning paths
- Create lifelong learning systems that allow career transitions as economies evolve

## Economic Domain
- Develop well-being metrics beyond GDP that measure health outcomes, ecological footprints, and happiness indicators
- Implement progressive taxation structures that fund publ